# Exam Hall Monitor: live scan on Colab

Upload a video and watch it being scanned while it plays: blurred heads, person IDs, skeletons,
phone boxes, and a line from each phone to the person it was matched to. People whose behaviour
lasts long enough get a red (phone) or orange (head turn, lean, out of seat) box, a flag in the
list under the video, and a short review clip.

**Every flag is a request for a human to review, never an accusation.**

**Before you start**
1. *Runtime → Change runtime type → T4 GPU*.
2. Copy the whole `exam-hall-monitor` folder into **Google Drive → My Drive** (or unzip
   `exam-hall-monitor.zip` there), so the path is `My Drive/exam-hall-monitor`.

**Privacy:** videos of real people go to your own Drive only. Use footage you have consent for,
and delete clips you no longer need.

In [ ]:
# 1. Check the GPU (should show a Tesla T4 or similar)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Mount Google Drive, find the project folder (unzip it if only the zip is there), go into it
import glob, os, zipfile
from google.colab import drive
drive.mount('/content/drive')

ROOT = '/content/drive/MyDrive'
def find(pattern):
    # look in My Drive and up to two folders deep
    for depth in ['', '*/', '*/*/']:
        hits = sorted(glob.glob(f'{ROOT}/{depth}{pattern}'))
        if hits:
            return hits[0]
    return None

req = find('exam-hall-monitor*/requirements.txt')
if req is None:
    # Google Drive does not unzip uploads, so unzip here
    zip_path = find('exam-hall-monitor*.zip')
    assert zip_path, 'No exam-hall-monitor folder or zip found in My Drive. Upload exam-hall-monitor.zip to My Drive first.'
    print('Unzipping', zip_path)
    zipfile.ZipFile(zip_path).extractall(os.path.dirname(zip_path))
    req = os.path.join(os.path.dirname(zip_path), 'exam-hall-monitor', 'requirements.txt')

PROJECT = os.path.dirname(req)
%cd {PROJECT}
!ls

In [ ]:
# 3. Install dependencies (PyTorch is already on Colab; takes ~1 minute)
!pip install -q -r requirements.txt

In [ ]:
# 3b. Quick self-check: the tests use synthetic data (no video, no models)
!python -m pytest -q tests

## 4. Choose a video
**Option A:** run the cell below and pick a file (fine for short clips; big files upload slowly).

**Option B (faster for big files):** drop the video into `My Drive/exam-hall-monitor/data/` in
Google Drive, skip the upload cell, and set `VIDEO` by hand in the next cell.

In [ ]:
# 4A. Upload a video from your computer into data/
import os, shutil
from google.colab import files

os.makedirs('data', exist_ok=True)
uploaded = files.upload()
name = next(iter(uploaded))
shutil.move(name, f'data/{name}')
VIDEO = f'data/{name}'
print('Video:', VIDEO)

In [ ]:
# 4B. Or point at a video already in data/ on Drive (uncomment and edit):
# VIDEO = 'data/my_clip.mp4'

In [ ]:
# 5. Scan it live. Frames appear below as they are processed.
#    Press the stop button on this cell to stop early; the video so far is still saved.
from src.utils import load_config
from src.live import play_in_notebook

cfg = load_config()
summary = play_in_notebook(VIDEO, cfg)

In [ ]:
# 6. Flags for HUMAN review: look at each clip and fill in review_decision yourself.
import pandas as pd
flags = pd.read_csv(summary['alerts_csv'])
flags

In [ ]:
# 6b. Metrics for this scan (no labels needed): speed, people per frame, alert rates,
#     baseline use, and precision once you have confirmed / dismissed the flags.
!python -m src.metrics --alerts {summary['alerts_csv']} --plot
from IPython.display import Image
Image(summary['alerts_csv'].replace('_alerts.csv', '_metrics.png'))


In [ ]:
# 7. Download the annotated video, the flags CSV and the review clips
#    (all of them are also saved in results/ on your Drive)
import shutil
from google.colab import files

files.download(summary['output'])
files.download(summary['alerts_csv'])
if len(flags):
    files.download(shutil.make_archive('/content/review_clips', 'zip', summary['clips_dir']))

## Evaluation for the report
Put labelled test clips in `data/eval/` and describe them in `data/labels.csv`
(format: see the top of `evaluate.py` or the README). Uncomment what you need.

In [ ]:
# Events: precision / recall / false alerts per hour, plus sweep, ablation and error gallery
# !python evaluate.py --clips-dir data/eval --labels data/labels.csv --sweep --ablation --gallery

# Phone detection modes (full / sahi / crop) on labelled phone images (YOLO folder or COCO JSON)
# !python evaluate.py --compare-modes --gt data/phone_set

## Optional: the Streamlit app (upload button in a web page)
This opens `app.py` in a new tab through Colab's own private proxy, so only your Google account
can reach it. If the tab stays blank, use the notebook player above instead; it shows the same thing.

In [ ]:
import subprocess, time
from google.colab import output

log = open('/content/streamlit.log', 'w')
streamlit_proc = subprocess.Popen(
    ['streamlit', 'run', 'app.py',
     '--server.port', '8501', '--server.headless', 'true',
     '--server.maxUploadSize', '1000',
     # Colab's proxy breaks Streamlit's CORS/XSRF checks; the proxy itself is private to you.
     '--server.enableCORS', 'false', '--server.enableXsrfProtection', 'false'],
    stdout=log, stderr=subprocess.STDOUT)
time.sleep(5)
output.serve_kernel_port_as_window(8501)
# To stop it later: streamlit_proc.terminate()